# Experimento: fuzzy somente no threshold

Este notebook compara o pipeline original com uma variante em que apenas o threshold do pre-processamento muda para fuzzy. Todas as etapas seguintes ficam iguais: deteccao de circulos, segmentacao da aorta, deteccao dos ostios, vesselness arterial, region growing e pos-processamento.

As saidas foram separadas por etapa para facilitar a leitura: carregamento, threshold, deteccao dos ostios, segmentacao arterial e resumo final.

## 1. Ambiente

Configura o caminho do repositorio e importa as funcoes usadas no experimento.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, NOTEBOOK_CWD.parent, NOTEBOOK_CWD.parent.parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()


In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.config_utils import scale_config_to_resolution
from utils.notebook_env import resolve_imagecas_base_path
from utils.config_utils import load_config_json
from utils.utils.nifti_io import load_raw_img_and_label
from utils.processing import (
    downscale_image_ndi,
    run_core_preprocessing_pipeline,
    threshold_image_with_offset,
)
from utils.segmentation import (
    build_lcc_image_from_mask,
    detect_and_evaluate_ostia,
    fuzzy_trapezoid_threshold,
    get_or_compute_vesselness,
    get_or_detect_aorta_circles,
    get_or_segment_aorta,
    segment_arteries_from_ostia,
)


## 2. Parametros do teste

Altere apenas esta celula para trocar a imagem, o fator de reducao ou a configuracao fuzzy. O cache fica desligado para evitar que resultados antigos influenciem a comparacao.

In [ ]:
CONFIG_PATH = REPO_ROOT / "config/pipeline_config.json"
BASE_PATH = resolve_imagecas_base_path()

IMG_ID = 458
DOWNSCALE_FACTORS = (2, 2, 1)
MIN_HU = -300

FUZZY_MARGIN_HU = 80
FUZZY_ALPHA_CUT = 0.50

SAVE_CSV_OUTPUTS = True
CSV_PREFIX = REPO_ROOT / "output/tmp_fuzzy_threshold_only"

CONFIG = load_config_json(str(CONFIG_PATH), {})
CONFIG["DOWNSCALE_FACTORS"] = list(DOWNSCALE_FACTORS)
CONFIG["LOAD_CACHE"] = False
CONFIG["SAVE_CACHE"] = False
RUN_CONFIG = scale_config_to_resolution(copy.deepcopy(CONFIG))

MAX_THRESHOLD_PERCENTILE = CONFIG.get("MAX_THRESHOLD_PERCENTILE", 99.7)

img_path = BASE_PATH / f"{IMG_ID}.img.nii.gz"
label_path = BASE_PATH / f"{IMG_ID}.label.nii.gz"

config_df = pd.DataFrame(
    [
        {
            "IMG_ID": IMG_ID,
            "base_path": str(BASE_PATH),
            "downscale_factors": DOWNSCALE_FACTORS,
            "max_threshold_percentile": MAX_THRESHOLD_PERCENTILE,
            "fuzzy_margin_hu": FUZZY_MARGIN_HU,
            "fuzzy_alpha_cut": FUZZY_ALPHA_CUT,
            "load_cache": RUN_CONFIG["LOAD_CACHE"],
            "save_cache": RUN_CONFIG["SAVE_CACHE"],
        }
    ]
)
config_df


## 3. Funcoes auxiliares

A funcao fuzzy gera um mapa de pertinencia entre 0 e 1. O `alpha-cut` transforma esse mapa em mascara binaria para manter o restante do pipeline igual ao original.

In [ ]:
def csv_path(step_name: str) -> Path:
    return CSV_PREFIX.with_name(f"{CSV_PREFIX.name}_{step_name}.csv")


def maybe_save_csv(df: pd.DataFrame, step_name: str) -> Path | None:
    if not SAVE_CSV_OUTPUTS:
        return None
    path = csv_path(step_name)
    df.to_csv(path, index=False)
    return path


In [ ]:
def run_detection_stage(
    variant_name: str,
    lcc_image: np.ndarray,
    label: np.ndarray,
    scaled_spacing: tuple[float, float, float],
    config: dict,
) -> dict:
    """Executa aorta + ostios. Nao cria cache quando SAVE_CACHE=False."""
    stage_root = REPO_ROOT / "output" / "tmp_fuzzy_threshold_only_no_cache"

    result = {
        "variant": variant_name,
        "ostia_found": False,
        "ostia_status": "not_evaluated",
        "both_correct": False,
        "both_tolerable": False,
        "left_dist_mm": np.inf,
        "right_dist_mm": np.inf,
        "left_intersects": False,
        "right_intersects": False,
        "ostia_left": None,
        "ostia_right": None,
        "num_circles": 0,
        "aorta_voxels": 0,
        "ostia_error": None,
        "error": None,
    }

    vesselness_ostios = get_or_compute_vesselness(
        str(IMG_ID),
        lcc_image,
        cache_dir=str(stage_root / variant_name / "vesselness_ostios_cache"),
        vesselness_config=config["VESSELNESS_AORTA"],
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
    )

    detected_circles = get_or_detect_aorta_circles(
        str(IMG_ID),
        lcc_image,
        DOWNSCALE_FACTORS,
        scaled_spacing,
        config["CIRCLE_DETECTION"],
        stage_root / variant_name,
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
        use_gpu=config.get("USE_GPU", False),
    )
    result["num_circles"] = len(detected_circles)

    aorta_mask = get_or_segment_aorta(
        str(IMG_ID),
        lcc_image,
        detected_circles,
        config["LEVEL_SET"],
        stage_root / variant_name,
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
        use_gpu=config.get("USE_GPU", False),
    )
    result["aorta_voxels"] = int(aorta_mask.sum())

    try:
        ostia_eval = detect_and_evaluate_ostia(
            aorta_mask,
            vesselness_ostios,
            label,
            scaled_spacing,
            config,
        )
    except ValueError as exc:
        result["ostia_status"] = "not_found"
        result["ostia_error"] = str(exc)
        return result

    if ostia_eval["both_correct"]:
        ostia_status = "both_correct"
    elif ostia_eval["both_tolerable"]:
        ostia_status = "both_tolerable"
    else:
        ostia_status = "found_but_wrong"

    result.update(
        {
            "ostia_found": True,
            "ostia_status": ostia_status,
            "both_correct": bool(ostia_eval["both_correct"]),
            "both_tolerable": bool(ostia_eval["both_tolerable"]),
            "left_dist_mm": ostia_eval["left_info"]["physical_dist"],
            "right_dist_mm": ostia_eval["right_info"]["physical_dist"],
            "left_intersects": ostia_eval["left_info"]["intersects"],
            "right_intersects": ostia_eval["right_info"]["intersects"],
            "ostia_left": tuple(map(int, ostia_eval["ostia_left"])),
            "ostia_right": tuple(map(int, ostia_eval["ostia_right"])),
            "label_artery": ostia_eval["label_artery"],
            "aorta_mask": aorta_mask,
            "vesselness_ostios": vesselness_ostios,
        }
    )
    return result


def run_segmentation_stage(
    detection_result: dict,
    lcc_image: np.ndarray,
    config: dict,
) -> dict:
    """Executa apenas a segmentacao arterial para casos com ostios encontrados."""
    result = {
        "variant": detection_result["variant"],
        "segmentation_attempted": False,
        "proceeded_with_bad_ostia": False,
        "dice_artery": 0.0,
        "artery_voxels": 0,
        "segmentation_error": None,
    }

    if not detection_result["ostia_found"]:
        result["segmentation_error"] = detection_result["ostia_error"]
        return result

    result["segmentation_attempted"] = True
    result["proceeded_with_bad_ostia"] = not (
        detection_result["both_correct"] or detection_result["both_tolerable"]
    )

    artery_metrics = segment_arteries_from_ostia(
        str(IMG_ID),
        lcc_image,
        detection_result["label_artery"],
        detection_result["ostia_left"],
        detection_result["ostia_right"],
        config,
        REPO_ROOT / "output" / "tmp_fuzzy_threshold_only_no_cache" / detection_result["variant"],
    )

    result["dice_artery"] = artery_metrics["dice_artery"]
    result["artery_voxels"] = int(artery_metrics["artery_voxels"])
    return result


## 4. Carregamento e reducao da imagem

Esta etapa carrega imagem/label, aplica o downsample na label e registra o espacamento usado nas metricas em milimetros.

In [ ]:
nii_img, nii_label = load_raw_img_and_label(str(img_path), str(label_path))
image = nii_img.get_fdata(dtype=np.float32)
label = nii_label.get_fdata(dtype=np.float32).astype(np.uint8)
spacing = tuple(float(value) for value in nii_img.header.get_zooms()[:3])

down_label = downscale_image_ndi(
    label,
    DOWNSCALE_FACTORS,
    order=0,
).astype(np.uint8)
scaled_spacing = tuple(
    spacing[idx] * DOWNSCALE_FACTORS[idx] for idx in range(len(DOWNSCALE_FACTORS))
)

sample_info_df = pd.DataFrame(
    [
        {
            "IMG_ID": IMG_ID,
            "image_shape": image.shape,
            "label_shape": label.shape,
            "down_label_shape": down_label.shape,
            "spacing_mm": spacing,
            "scaled_spacing_mm": scaled_spacing,
        }
    ]
)
maybe_save_csv(sample_info_df, "sample_info")
sample_info_df


## 5. Threshold

Aqui ficam apenas as saidas do pre-processamento. A linha `baseline_original_threshold` usa o threshold crisp original. A linha `fuzzy_threshold_only` usa a mesma faixa central, mas expande a borda pela pertinencia fuzzy e pelo `alpha-cut`.

In [ ]:
down_image, _, baseline_lcc_image, baseline_threshold_values = run_core_preprocessing_pipeline(
    image,
    downscale_factors=DOWNSCALE_FACTORS,
    min_threshold=MIN_HU,
    max_threshold_percentile=MAX_THRESHOLD_PERCENTILE,
    lcc_per_slice=True,
    order=3,
    use_opencv=False,
)

max_hu = float(baseline_threshold_values[1])
fuzzy_membership = fuzzy_trapezoid_threshold(
    down_image,
    min_hu=MIN_HU,
    max_hu=max_hu,
    margin_hu=FUZZY_MARGIN_HU,
)
fuzzy_mask = fuzzy_membership >= FUZZY_ALPHA_CUT
fuzzy_effective_min_hu = MIN_HU - FUZZY_MARGIN_HU * (1 - FUZZY_ALPHA_CUT)
fuzzy_effective_max_hu = max_hu + FUZZY_MARGIN_HU * (1 - FUZZY_ALPHA_CUT)
fuzzy_lcc_image, fuzzy_lcc_mask = build_lcc_image_from_mask(
    down_image,
    fuzzy_mask,
    offset=abs(MIN_HU),
    per_slice=True,
)

_, baseline_mask, _ = threshold_image_with_offset(
    down_image,
    min_val=MIN_HU,
    max_val=int(max_hu),
)
_, baseline_lcc_mask = build_lcc_image_from_mask(
    down_image,
    baseline_mask,
    offset=abs(MIN_HU),
    per_slice=True,
)

threshold_df = pd.DataFrame(
    [
        {
            "variant": "baseline_original_threshold",
            "threshold_mode": "crisp",
            "core_min_hu": MIN_HU,
            "core_max_hu": max_hu,
            "effective_min_hu": MIN_HU,
            "effective_max_hu": max_hu,
            "fuzzy_margin_hu": np.nan,
            "fuzzy_alpha_cut": np.nan,
            "threshold_voxels": int(baseline_mask.sum()),
            "lcc_voxels": int(baseline_lcc_mask.sum()),
        },
        {
            "variant": "fuzzy_threshold_only",
            "threshold_mode": "fuzzy_alpha_cut",
            "core_min_hu": MIN_HU,
            "core_max_hu": max_hu,
            "effective_min_hu": fuzzy_effective_min_hu,
            "effective_max_hu": fuzzy_effective_max_hu,
            "fuzzy_margin_hu": FUZZY_MARGIN_HU,
            "fuzzy_alpha_cut": FUZZY_ALPHA_CUT,
            "threshold_voxels": int(fuzzy_mask.sum()),
            "lcc_voxels": int(fuzzy_lcc_mask.sum()),
        },
    ]
)
maybe_save_csv(threshold_df, "threshold")
threshold_df


## 6. Deteccao da aorta e dos ostios

Esta etapa mostra apenas o que vem antes da segmentacao arterial: circulos detectados, tamanho da mascara da aorta, status dos ostios e distancias ate a label.

In [ ]:
variant_inputs = {
    "baseline_original_threshold": baseline_lcc_image,
    "fuzzy_threshold_only": fuzzy_lcc_image,
}

detection_results = {
    variant: run_detection_stage(
        variant,
        lcc_image,
        down_label,
        scaled_spacing,
        RUN_CONFIG,
    )
    for variant, lcc_image in variant_inputs.items()
}

ostia_df = pd.DataFrame(
    [
        {
            "variant": result["variant"],
            "ostia_status": result["ostia_status"],
            "ostia_found": result["ostia_found"],
            "both_correct": result["both_correct"],
            "both_tolerable": result["both_tolerable"],
            "left_dist_mm": result["left_dist_mm"],
            "right_dist_mm": result["right_dist_mm"],
            "ostia_left": result["ostia_left"],
            "ostia_right": result["ostia_right"],
            "num_circles": result["num_circles"],
            "aorta_voxels": result["aorta_voxels"],
            "ostia_error": result["ostia_error"],
        }
        for result in detection_results.values()
    ]
)
maybe_save_csv(ostia_df, "ostia_detection")
ostia_df


## 7. Segmentacao arterial

Esta etapa roda apenas a segmentacao arterial a partir dos ostios encontrados. Se os ostios nao forem encontrados, a segmentacao e marcada como nao executada.

In [ ]:
segmentation_results = {
    variant: run_segmentation_stage(
        detection_result,
        variant_inputs[variant],
        RUN_CONFIG,
    )
    for variant, detection_result in detection_results.items()
}

segmentation_df = pd.DataFrame(segmentation_results.values())
maybe_save_csv(segmentation_df, "segmentation")
segmentation_df


## 8. Resumo final

Resumo compacto com os campos mais importantes para comparar o baseline com o threshold fuzzy.

In [ ]:
summary_df = (
    threshold_df[
        [
            "variant",
            "threshold_mode",
            "effective_min_hu",
            "effective_max_hu",
            "threshold_voxels",
            "lcc_voxels",
        ]
    ]
    .merge(
        ostia_df[
            [
                "variant",
                "ostia_status",
                "both_correct",
                "both_tolerable",
                "left_dist_mm",
                "right_dist_mm",
                "num_circles",
                "aorta_voxels",
            ]
        ],
        on="variant",
    )
    .merge(
        segmentation_df[
            [
                "variant",
                "segmentation_attempted",
                "dice_artery",
                "artery_voxels",
                "proceeded_with_bad_ostia",
            ]
        ],
        on="variant",
    )
)

maybe_save_csv(summary_df, "summary")
summary_df


## 9. Visualizacao do threshold

Figura focada no pre-processamento. Ela nao mistura as metricas de ostios ou segmentacao.

In [ ]:
z_idx = down_image.shape[2] // 2
vmin, vmax = np.percentile(down_image[:, :, z_idx], [1, 99])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0, 0].imshow(down_image[:, :, z_idx], cmap="gray", vmin=vmin, vmax=vmax)
axes[0, 0].set_title("Imagem reduzida")

axes[0, 1].imshow(baseline_mask[:, :, z_idx], cmap="gray")
axes[0, 1].set_title("Threshold crisp")

axes[0, 2].imshow(fuzzy_membership[:, :, z_idx], cmap="magma", vmin=0, vmax=1)
axes[0, 2].set_title("Pertinencia fuzzy")

axes[1, 0].imshow(baseline_lcc_image[:, :, z_idx], cmap="gray")
axes[1, 0].set_title("LCC crisp")

axes[1, 1].imshow(fuzzy_lcc_image[:, :, z_idx], cmap="gray")
axes[1, 1].set_title("LCC fuzzy")

axes[1, 2].imshow(fuzzy_mask[:, :, z_idx], cmap="gray")
axes[1, 2].set_title("Mascara alpha-cut")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.show()
